### fkl topk

记 $\mathcal K_t=\text{top-}k\big(p_t(\cdot\mid y_{<t})\big)$ 是教师在位置 $t$ 的 top-k token 集合（代码里的 teacher_topk_ids，$k$ 默认 64）。逐 token 损失是：

$$
\ell_t(\theta)=\sum_{v\in\mathcal{K}_t}p_t(v)\left[\log p_t(v)-\log q_\theta\!\left(v\mid y_{<t}\right)\right]
$$

- `fsdp/losses.py`

```python
student_log_probs = F.log_softmax(student_logits, dim=-1)                    # 全词表归一化
student_topk_log_probs = torch.gather(student_log_probs, -1, teacher_topk_ids)  # 在教师的 id 上 gather
...
kld = p * (log_p - log_q);  return kld.sum(dim=-1)                           # 只对 K_t 求和
```

$$
M_t^{\mathrm{teacher}}=\sum_{v\in\mathcal{K}_t}p_t(v)<1,\qquad M_t^{\mathrm{student}}=\sum_{v\in\mathcal{K}_t}q_\theta(v)<1
$$

举例，某位置教师给某 token $p=0.1=10^{-1}$，学生给 $q=10^{-4}$（8B 学生 vs 32B 教师，蒸馏早期完全正常）。

| | 数值 |
|---|---|
| $w$ | $10^3$ |
| $w\log w$（命中时的估计值） | $\approx 6900$ |
| 命中概率 | $10^{-4}$ |
| 对均值的贡献  | $10^{-4}\times 6900=0.69$ ✓（$=p\log w$，无偏） |
| 对二阶矩的贡献 | $10^{-4}\times 6900^2\approx 4.8\times10^3$ |
| 相对标准差 $\mathrm{RSD}=\sigma/\mu$ | $\sqrt{4761}/0.69\approx \mathbf{100\times}$ |

$q=10^{-4}$ 估计值是 0，剩下那一次是 6900。要把相对误差压到 1，得每个位置采 $\sim10^4$ 个样本。而 reverse KL 的 k1 在同一个位置是 $\log q-\log p$，量级 O(1)~O(10)，有界、无权重。$p\gg q$ 处高方差

```python
if self.use_policy_gradient and self.loss_mode == "forward_kl_topk":
    print("WARNING: forward_kl_topk is most effective as a supervised distillation loss (use_policy_gradient=False). With policy gradient, the update uses only the sampled token's logprob ∇logπ(a), so the top-k distributional signal (how non-sampled logits should move) is largely unused.")
```

- 在同一个生成位置 $t$，teacher 给出的 top-k 候选中，没有被 student rollout 实际选为 $y_t$ 的那些 token。

| 对比项 | `use_pg=False`（直接反传） | `use_pg=True`（走 PG） |
|---|---|---|
| token loss | $\ell_t=\displaystyle\sum_{v\in\mathcal{K}_t}p_t(v)\big[\log p_t(v)-\log q_\theta(v\mid y_{<t})\big]$ | $\ell_t^{\mathrm{PG}}=-\min\big(\rho_t A_t,\ \mathrm{clip}(\rho_t,1-\epsilon,1+\epsilon),A_t\big)$，$A_t=-\mathrm{sg}[\ell_t]$ |
| logit 梯度 | $M_t q_\theta(u)-p_t(u)\mathbb{1}[u\in\mathcal{K}_t]$ | $-A_t\rho_t\bigl(\mathbb{1}[u=y_t]-q_\theta(u)\bigr)$ |
| 教师信息进入方式 | $k$ 个系数 $\{p_t(v)\}_{v\in\mathcal{K}_t}$，直接进入梯度 | 一个标量 $A_t$，只进入梯度系数 |
| 非采样 token 的方向 | 每个 token 分别按照 $p_t(v)$ 显式顶上去 | 只受 softmax 归一化项影响而被统一压低，更新方向不包含各 token 的教师信息 |
| 每个位置的自由度 | $64$ | $1$ |

- $m_K=\sum_{v\in \mathcal K_t}p_v$ (teacher mass)